**RAG implementation on a story**

So first, I saved the story in a txt file and used TextLoader to make it a Document data type. Then I used RecursiveCharacterTextSplitter to divide the document into various number of chunks, so that its easier for searching.

I then embedded the chunks using an open source model "sentence-transformers" and stored in a vector db called chromadb.

I connected the Gemini LLM and created a prompt and gave it context about the story related to the question. For example if the question is how old is the character named Elias, the context retrieved will be related to the question. This is done by using embeddings and cosine-similarity.

The LLM then appropriately answers the question.


In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf

In [2]:
from langchain_core.documents import Document

In [3]:
# from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/content/the_lanterns_of_greybridge.txt",encoding = "utf-8")

document = loader.load()
document

[Document(metadata={'source': '/content/the_lanterns_of_greybridge.txt'}, page_content='The Lanterns of Greybridge\nIn the quiet riverside town of Greybridge, there stood an old bookshop called Hollow & Pine, squeezed between a bakery and a watch repair store on Maple Street. The shop belonged to Elias Moreau, a sixty-three-year-old man known for keeping handwritten records of every customer who had visited since 1987.\n\nElias lived above the shop with his orange cat, Bristle, who spent most afternoons sleeping near the cracked window overlooking the river. Every morning at exactly 6:15 a.m., Elias opened the bookstore, swept the entrance, and brewed dark coffee in a copper kettle inherited from his grandmother, Clara Moreau.\n\nOne rainy Tuesday in October 2019, a university student named Mira Solis entered the shop while searching for books about local folklore. Mira was twenty-two, studying historical anthropology at North Valley University, and had recently moved into Apartment 3B

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # max characters per chunk
    chunk_overlap=100    # overlap between chunks
)

chunks = text_splitter.split_documents(document)

print(len(chunks))
print(chunks[2].page_content)
print(chunks[2].metadata)


35
between a bakery and a watch repair store on Maple Street. The shop belonged to Elias Moreau, a sixty-three-year-old man known for keeping handwritten records of every customer who had visited since
{'source': '/content/the_lanterns_of_greybridge.txt'}


In [5]:
!pip install chromadb sentence-transformers langchain-chroma

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc.page_content for doc in chunks]
embeddings = model.encode(texts).tolist()

print(embeddings[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[-0.0020343230571597815, 0.07911525666713715, 0.012202024459838867, 0.03199096769094467, -0.010415558703243732, -0.0058420561254024506, 0.08233914524316788, -0.07587148249149323, -0.04579350724816322, -0.0376087911427021, 0.011279767379164696, -0.06967972218990326, -0.03490918502211571, 0.021185530349612236, -0.060046467930078506, 0.036147452890872955, -0.04491132125258446, -0.009526925161480904, -0.06491947919130325, -0.016395369544625282, -0.0014471408212557435, 0.02411862649023533, -0.034432731568813324, 0.018200282007455826, -0.03633451461791992, 0.039815545082092285, -0.045744236558675766, -0.004619273357093334, 0.004719124175608158, -0.07565165311098099, -0.020345481112599373, -0.02517939731478691, -0.09719444066286087, 0.0038551499601453543, 0.029082970693707466, -0.04394685477018356, 0.034981224685907364, 0.05856704339385033, 0.03418523445725441, -0.019978690892457962, -0.008997797034680843, -0.06901261955499649, -0.030689168721437454, 0.018637273460626602, -0.06821273267269135

In [8]:

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection("story")

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=[str(i) for i in range(len(texts))]
)

print("Stored")

Stored


In [29]:
def retrieve_documents(query):

  results = collection.query(
      query_embeddings=[query],
      n_results=3
  )

  return results

In [14]:
!pip install google-generativeai

In [35]:
import google.generativeai as genai
from sentence_transformers import SentenceTransformer


genai.configure(api_key="n0t_r34l_4pi_k3y")

model = genai.GenerativeModel("gemini-2.5-flash")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device = "cpu")


def ask_rag(question):
    # Embed query
    query_embedding = embedding_model.encode(question).tolist()

    # Retrieve chunks
    results = retrieve_documents(query_embedding)
    print(results["documents"])

    context = "\n\n".join(results["documents"][0])

    # Prompt Gemini
    prompt = f"""
Answer the user's question using ONLY the context below.

Context:
{context}

Question:
{question}

If the answer is not present, say:
"I couldn't find that in the document." And then explain why you felt that way.
"""

    response = model.generate_content(prompt)
    print()
    return response.text



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:

print(ask_rag("what was eliass wifes name? how did she die?"))

[['Elias quietly placed a lantern for his late wife, Isabelle Moreau, who had died in 2008 after a long illness. Daniel brought one for his grandfather Arthur Reeves, a volunteer rescuer during the 1968', 'Elias lived above the shop with his orange cat, Bristle, who spent most afternoons sleeping near the cracked window overlooking the river. Every morning at exactly 6:15 a.m., Elias opened the', 'in almost fifty years. Mira, Daniel, Elias, Sofia, and Marcus stood near the old bridge while children carried paper lanterns painted with names and memories.']]

Elias's wife's name was Isabelle Moreau. She died after a long illness.
